# Solveit Auth Service Demo

Solveit auth service makes it easy to use 'Sign In With Google' for apps hosted on solveit without all the fuss.

This minimal demo shows how you can create a sign in link, which will send the user to google, and thence to auth.solve.it.com, and eventually back to your app at a specified callback URL. At which point you'll be able to access their sub ID, but no other information about them. This is a unique identifier, and can be used to store user-specific state in your database and so on. In this demo, the callback stores this id in the session, and uses it to gate a secret page so that only those who are signed in can see it. 

Author: Johno. Purpose: showcase solveit auth service. State: minimal first pass

## Setup

In [ ]:
import os, json, jwt as pyjwt, httpx
from fasthtml.common import *
from fasthtml.jupyter import *

In [ ]:
AUTH_URL = 'https://auth.solve.it.com.ai/request_signin'
KEY = os.environ['AAI_USER_KEY']
APP_URL = f'https://{json.loads(os.environ["PUBLIC_DOMAINS"])["8000"]}.solve.it.com'
CALLBACK = APP_URL+'/signin_completed'

In [ ]:
app, rt = fast_app()
srv = JupyUvi(app, port=8000)

In [ ]:
@rt
def index():
    return Titled('Auth Test', A('Sign In with Google', href='/do_signin', cls='button'))

@rt
def secret(session):
    if not 'auth' in session: return "ACCESS DENIED"
    return Titled('Secret page', "Congrats! You are signed in")

## The sign in routes

do_signin initializes the process

signin_completed is the callback URL, and recieves a payload with the `signin_reply` that comes from solveit's auth service. Here is how to parse that and get the user sub id:

In [ ]:
@rt
def do_signin():
    r = httpx.post(AUTH_URL, headers={'Authorization': KEY}, json={'callback_url': CALLBACK, "email_re": ".*@answer.ai"})
    return RedirectResponse(r.json()['signin_url'])

@rt
def signin_completed(session, signin_reply: str = ''):
    try:
        payload = pyjwt.decode(signin_reply, KEY, algorithms=['HS256'])
        if 'err' in payload: raise Exception(payload['err'])
        sub_id = str(payload['sub'])
        session['auth'] = sub_id # In other routes, we can check for this
        return Titled('Signed In!', Pre('Sub ID:'+ sub_id))
    except Exception as e:
        return Title('Error'), Main(H1('Auth Failed'), Pre(str(e)))

## Testing it out

Visit this link and log in - if all goes well, you should then be able to check out `/secret`.

In [ ]:
A("Try it out", href=APP_URL, target='_blank')

<a href="https://awesome-mirror-achieves-3cpc05.hacksi.answer.ai" target="_blank">Try it out</a>

## Signing out

Popping the auth key from the session is an easy way to log out.

In [ ]:
@rt
def sign_out(session):
    session.pop('auth', "")
    return 'Signed out'

## Going further

From here, some things you may wish to explore:

- Using Beforeware to gate many/all routes with auth
- Setting up a database to persist state/info per user, with `sub` as the primary key
- Passing the optional `email_re` parameter to make it so that only certain people can log in. (e.g. `json={'callback_url': CALLBACK, "email_re": ".*@answer.ai"}` will give `auth_failed` if a non-answerai email address is used.